# 01 — Data collection from Bluesky


## 1. Login


Here I am logging in to my bluesky account with my handle and the app password that I have created.

In [3]:
import json
from atproto import Client

creds = json.load(open("../../secrets/bluesky_credentials.json"))
client = Client()
client.login(creds["HANDLE"], creds["APP_PASSWORD"])
print("Logged in as:", client.me.handle)

Logged in as: egeaydemir.bsky.social


## 2. Collection helpers


Defines the collection helpers: a retry wrapper for rate limits, a facet extractor for hashtags,mentions,links and a paginated searchPosts collector that returns one row per post within a fixed time window.

Update (added while testing the thread fetching): I noticed that `call_with_retries` also retried 4xx errors. When a post has been deleted, `getPostThread` returns 400 NotFound, and retrying that 10 times with growing waits costs about two minutes per deleted post for nothing. Now only 429 (rate limit), 5xx and network errors are retried; other 4xx errors are raised immediately and counted as failed threads.

In [9]:
import time
from datetime import datetime, timezone
import pandas as pd
from atproto import models
from atproto_client.exceptions import RequestException, InvokeTimeoutError, NetworkError


def parse_dt_utc(iso_str):
    #Here we are putting all the posts into the same timeframe (UTC)
    dt = datetime.fromisoformat(iso_str.replace("Z", "+00:00"))
    return dt.replace(tzinfo=timezone.utc) if dt.tzinfo is None else dt.astimezone(timezone.utc)


def call_with_retries(fn, *args, max_retries=10, **kwargs):
    #This function works like an insurance to us. If we hit a  rate limit more than 10 times our collection of posts might stop before finishing.
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except (InvokeTimeoutError, NetworkError):
            wait = min(60, 2 ** attempt)
            print(f"[network] sleeping {wait}s")
            time.sleep(wait)
        except RequestException as e:
            resp = getattr(e, "response", None)
            status = getattr(resp, "status_code", None)
            headers = getattr(resp, "headers", {}) or {}
            if status == 429:
                reset = headers.get("ratelimit-reset") or headers.get("RateLimit-Reset")
                wait = max(1, int(reset) - int(time.time())) + 1 if reset else min(60, 2 ** attempt)
                print(f"[429] rate limited, sleeping {wait}s")
            elif status is not None and 400 <= status < 500:
                #Client errors like a deleted post (400 NotFound) can not be fixed by retrying, so we raise them immediately instead of waiting 10 times.
                raise
            else:
                wait = min(20, 2 ** attempt)
                print(f"[{status}] request failed, sleeping {wait}s")
            time.sleep(wait)
    raise RuntimeError("Too many retries")


def extract_facets(record):
    #We use DID instead of handles for making our mention network. Since all hasthags, mentions and links (facets) are stored seperate from text
    hashtags, mentions, links = [], [], []
    for facet in getattr(record, "facets", None) or []:
        for feat in getattr(facet, "features", None) or []:
            ftype = getattr(feat, "py_type", None)
            if ftype == "app.bsky.richtext.facet#tag":
                hashtags.append(feat.tag)
            elif ftype == "app.bsky.richtext.facet#mention":
                mentions.append(feat.did)
            elif ftype == "app.bsky.richtext.facet#link":
                links.append(feat.uri)
    return {"hashtags": hashtags, "mentions": mentions, "links": links}


def collect_search_posts(client, q, since_iso=None, until_iso=None,
                         limit_total=200, page_size=100, polite_sleep=0.6, verbose=True):
    #Page size is 100 wrt  maximum endpoint that we have. Since we are only able to get cloose to 3.000 request per 5 minutes, I have choosen 0.6 seconds of pause between requests.
    rows, cursor, page = [], None, 0
    since_dt = parse_dt_utc(since_iso) if since_iso else None
    until_dt = parse_dt_utc(until_iso) if until_iso else None

    while len(rows) < limit_total:
        page += 1
        params = models.AppBskyFeedSearchPosts.Params(
            q=q, sort="latest", since=since_iso, until=until_iso,
            limit=min(page_size, limit_total - len(rows)), cursor=cursor,
        )
        res = call_with_retries(client.app.bsky.feed.search_posts, params)
        cursor, posts = res.cursor, res.posts or []
        if not posts:
            break

        for p in posts:
            rec = p.record
            created = parse_dt_utc(rec.created_at)
            if (since_dt and created < since_dt) or (until_dt and created >= until_dt):
                continue
            f = extract_facets(rec)
            reply = getattr(rec, "reply", None)
            embed = getattr(rec, "embed", None)
            rows.append({
                "uri": p.uri, "cid": p.cid,
                "created_at": created, "text": rec.text,
                "author_handle": p.author.handle, "author_did": p.author.did,
                "author_display_name": p.author.display_name,
                "reply_count": p.reply_count, "repost_count": p.repost_count,
                "like_count": p.like_count, "quote_count": p.quote_count,
                "hashtags": f["hashtags"], "mentions": f["mentions"], "links": f["links"],
                "langs": getattr(rec, "langs", None),
                "reply_root_uri": reply.root.uri if reply else None,
                "reply_parent_uri": reply.parent.uri if reply else None,
                "embed_type": getattr(embed, "py_type", None) if embed else None,
                "query": q,
            })
            if len(rows) >= limit_total:
                break

        if verbose:
            print(f"[page {page}] collected={len(rows)} oldest={min(r['created_at'] for r in rows):%Y-%m-%d %H:%M}")
        if cursor is None:
            break
        time.sleep(polite_sleep)

    return pd.DataFrame(rows)

## 3. Volume check 


I want to check if the function works, with 200 samples. And how many days does it took for checking the daily volume and lastly to check if the thread is really about finance.

In [5]:
df_test = collect_search_posts(
    client,
    q='"stock market" lang:en',
    since_iso="2026-08-13T00:00:00.000Z",
    until_iso="2026-09-13T00:00:00.000Z",
    limit_total=200,
)

print(df_test.shape)
print(df_test["created_at"].min(), "→", df_test["created_at"].max())
print("share of replies:", df_test["reply_parent_uri"].notna().mean().round(2))
df_test[["created_at", "author_handle", "reply_count", "like_count", "text"]].head(10)

[page 1] collected=98 oldest=2026-09-12 05:14
[page 2] collected=198 oldest=2026-09-11 16:53
[page 3] collected=200 oldest=2026-09-11 16:51
(200, 19)
2026-09-11 16:51:49.918000+00:00 → 2026-09-12 23:39:14.482000+00:00
share of replies: 0.5


,created_at,author_handle,reply_count,like_count,text
0,2026-09-12 23:39:14.482000+00:00,maxmoranhi.bsky.social,0,32,"""THE MACHINES COULD EXTERMINATE ALL HUMAN LIFE..."
1,2026-09-12 23:32:07.257949+00:00,newsanalysis.com,0,0,Nvidia is reportedly in talks to bankroll Anth...
2,2026-09-12 23:13:17.877000+00:00,blobodelendaest.bsky.social,1,1,The only direct link of 9/11 and the US collap...
3,2026-09-12 23:00:04.670182+00:00,newshoundai.bsky.social,1,0,Financial terms of the agreement were not disc...
4,2026-09-12 22:35:39.185000+00:00,todders707.bsky.social,0,0,I think they see the handwriting on the wall w...
5,2026-09-12 22:21:29.439000+00:00,wooderson69.bsky.social,0,4,AI probably figuring out a way to manipulate t...
6,2026-09-12 21:54:49.974000+00:00,karliechronicles.bsky.social,0,0,"It's ok, I'm sure the stock market will find a..."
7,2026-09-12 21:48:12.475000+00:00,nimakokaaz.bsky.social,1,5,this is why leftists don't invest in the stock...
8,2026-09-12 21:42:50+00:00,mueckinvest.bsky.social,0,0,📘 Briefly explained\r\nThe stock market is the...
9,2026-09-12 21:34:07.442000+00:00,tenej-jen.bsky.social,0,3,Yet another bubble to exploit the stock market...


This was a test and even from a test we got some insights about the posts, %50 of the collected posts are themselves replies to another post (share of replies: 0.5), so there really is a chat and its possible to create a network.

Before the real collection I want to see which search terms actually work. I try 12 candidate
terms in three groups (equities, macro/Fed, crypto) on the same 3-day window (9–11 Sep) and
count posts per day, share of replies and unique authors for each. I will pick the final
queries and the sub-streams to compare based on this table.

In [6]:
candidate_queries = {
    "equities": ['"stock market" lang:en', '"Wall Street" lang:en', '"S&P 500" lang:en', 'Nasdaq lang:en'],
    "macro":    ['"the Fed" lang:en', 'FOMC lang:en', '"interest rates" lang:en', '"rate cut" lang:en', 'Powell lang:en'],
    "crypto":   ['bitcoin lang:en', 'crypto lang:en', 'ethereum lang:en'],
}

rows = []
for stream, queries in candidate_queries.items():
    for q in queries:
        df_q = collect_search_posts(
            client, q,
            since_iso="2026-09-09T00:00:00.000Z",
            until_iso="2026-09-12T00:00:00.000Z",
            limit_total=5000, verbose=False,
        )
        rows.append({
            "stream": stream, "query": q,
            "posts_3d": len(df_q),
            "posts_per_day": round(len(df_q) / 3),
            "reply_share": round(df_q["reply_parent_uri"].notna().mean(), 2) if len(df_q) else None,
            "unique_authors": df_q["author_did"].nunique() if len(df_q) else 0,
        })
        print(f"{stream:9s} {q:28s} {len(df_q):5d} posts")

volume = pd.DataFrame(rows)
volume

equities  "stock market" lang:en         636 posts
equities  "Wall Street" lang:en         1550 posts
equities  "S&P 500" lang:en              215 posts
equities  Nasdaq lang:en                 655 posts
macro     "the Fed" lang:en              619 posts
macro     FOMC lang:en                    64 posts
macro     "interest rates" lang:en       866 posts
macro     "rate cut" lang:en              16 posts
macro     Powell lang:en                 669 posts
crypto    bitcoin lang:en               1332 posts
crypto    crypto lang:en                5000 posts
crypto    ethereum lang:en               257 posts


,stream,query,posts_3d,posts_per_day,reply_share,unique_authors
0,equities,"""stock market"" lang:en",636,212,0.45,519
1,equities,"""Wall Street"" lang:en",1550,517,0.32,1190
2,equities,"""S&P 500"" lang:en",215,72,0.13,135
3,equities,Nasdaq lang:en,655,218,0.05,154
4,macro,"""the Fed"" lang:en",619,206,0.48,481
5,macro,FOMC lang:en,64,21,0.33,40
6,macro,"""interest rates"" lang:en",866,289,0.31,696
7,macro,"""rate cut"" lang:en",16,5,0.38,15
8,macro,Powell lang:en,669,223,0.34,504
9,crypto,bitcoin lang:en,1332,444,0.22,532


Here we learn the volumes are really different from eachother, for example crypto fot hit by our 5000 limit (its really wide), on the other hand "rate cut" and "FOMC" have really low volumes, but volume can be misleading thats why we are checking for reply_share and unique autors(for bot checking). "the Fed" and "stock market" really have high reply shares so they really chat but Nasdaq has 655 post but only have 0.05 reply share so I have evaluated this as a automatic posts. And the unique authors table shows us that in Nasdaq 655 posts only have 154 unique authors so we can conclude that there are some bots in there.

## 4. Suspicious Queries
Numbers are not optimal by themsleves so here I am checking the queries of the post texts and the active accounts for to see what the queries gave us. (Just a reminder here I am using Warsh instead of Powell because I forgot that the FED chairman changed.)

In [7]:
for q in ['Nasdaq lang:en', '"Wall Street" lang:en', 'bitcoin lang:en', 'Warsh lang:en']:
    df_q = collect_search_posts(client, q, since_iso="2026-09-09T00:00:00.000Z",
                                until_iso="2026-09-12T00:00:00.000Z", limit_total=300, verbose=False)
    print(f"\n=== {q}: {len(df_q)} posts, {df_q['author_did'].nunique()} authors")
    print("most active:", df_q["author_handle"].value_counts().head(3).to_dict())
    for t in df_q.sample(5, random_state=1)["text"]:
        print("  -", t[:110].replace("\n", " "))


=== Nasdaq lang:en: 300 posts, 79 authors
most active: {'liquiditynews.bsky.social': 122, 'insyn.tommie.nu': 25, 'tradingstats.xyz': 24}
  - #PRSO $PRSO  Peraso (NASDAQ: PRSO): Tactical Autonomy and Stealth Communications Target Next-Generation Defens
  - Trading Halted for $RFAI  RF Acq Cp II Ord | at 10:16:58.836 ET on NASDAQ | Volatility Trading Pause | 2026-09
  - There were no companies in the S&P 500 & NASDAQ 100 that developed a Death Cross today.
  - @bsky.app 📈 Nasdaq is putting $100M into Kraken parent Payward at a reported $21B valuation. The partnership a
  - Trading Resumed for $TNON  Tenon Medical Inc Cm | at 10:43:50 ET on NASDAQ | 2026-09-11

=== "Wall Street" lang:en: 300 posts, 266 authors
most active: {'rosepatriots.bsky.social': 8, 'marketwire.bsky.social': 5, 'headline-feed.bsky.social': 4}
  - Presse-citron: #ChatGPT s'attaque aux métiers de la #finance : l' #analyste qui ne dort jamais est déjà validé
  - www.levernews.com/trumps-about...
  - . The Wall Street J

**What I saw.** Volume alone is misleading. Some terms are big but not conversational:
Nasdaq has 218 posts/day but only 4% replies and 4 posts per author one account
liquiditynews posts automated "Trading Halted/Resumed" alerts. Wall Street is large
and conversational but off-topic: the sample is mostly non-English posts that slipped through lang:en, generic "Wall Street
Journal" citations, and posts about AI or politics rather than the market itself. crypt` hit the 5,000 cap in
3 days and is mostly promotion. rate cut FOMC are small; since Powell is not the FED president Wash is the right choice

**Selection criteria.** (1) topical precision (sample inspection), (2) conversation:
share of replies ≥ ~0.3, (3) author diversity: ≤ ~1.5 posts per author in 3 days,
(4) enough volume (≥ ~50 posts/day per stream).

**Final set (all `lang:en`, 13 Aug – 12 Sep 2026):**
- *equities*: `"stock market"`, `"S&P 500"`
- *macro*: `"the Fed"`, `Warsh`, `FOMC`
- *crypto*: `bitcoin`, `ethereum`

**Excluded:** `"Wall Street"` (off-topic noise), `Nasdaq` (automated feeds), `crypto`
(too broad/spam), `Powell` (former chair, redundant with `Warsh`), `"rate cut"` (too small),
`"interest rates"` (broad: mortgages, other countries; would double the macro stream with
lower precision).

**Note for later.** `bitcoin`/`ethereum` keep some automated accounts (signal bots, hourly
stats). I keep them at collection time and flag them in preprocessing instead of filtering
by hand here. Already visible: the crypto stream is much less conversational (22% replies)
than equities/macro (~45%).

## 5. Acquisition
Using the 7 selected queries, I fetch all posts from August 13 to September 12, remove duplicates, and write the raw post file to disk.

In [8]:
from datetime import datetime, timezone
import os

final_queries = {
    "equities": ['"stock market" lang:en', '"S&P 500" lang:en'],
    "macro":    ['"the Fed" lang:en', 'Warsh lang:en', 'FOMC lang:en'],
    "crypto":   ['bitcoin lang:en', 'ethereum lang:en'],
}
SINCE, UNTIL = "2026-08-13T00:00:00.000Z", "2026-09-13T00:00:00.000Z"
RAW_DIR = "../data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

frames = []
for stream, queries in final_queries.items():
    for q in queries:
        df_q = collect_search_posts(client, q, since_iso=SINCE, until_iso=UNTIL,
                                    limit_total=50_000, verbose=False)
        df_q["stream"] = stream
        frames.append(df_q)
        pd.concat(frames, ignore_index=True).to_pickle(f"{RAW_DIR}/search_raw.pkl")   # checkpoint
        print(f"{stream:9s} {q:26s} {len(df_q):6d} posts  oldest={df_q['created_at'].min():%Y-%m-%d}")

raw = pd.concat(frames, ignore_index=True)
raw["collected_at"] = datetime.now(timezone.utc)

# one row per post; remember every query/stream that matched it
membership = raw.groupby("uri").agg(queries=("query", lambda s: sorted(set(s))),
                                    streams=("stream", lambda s: sorted(set(s)))).reset_index()
posts = (raw.drop_duplicates("uri")
            .drop(columns=["query", "stream"])
            .merge(membership, on="uri"))

print("\n--- funnel ---")
print("rows returned by all queries :", len(raw))
print("unique posts (by uri)        :", len(posts))
print("posts in >1 stream           :", (posts["streams"].str.len() > 1).sum())
print("posts per stream             :", posts["streams"].explode().value_counts().to_dict())
print("date range                   :", posts["created_at"].min(), "→", posts["created_at"].max())

posts.to_pickle(f"{RAW_DIR}/posts_search.pkl")
posts.to_json(f"{RAW_DIR}/posts_search.jsonl", orient="records", lines=True, date_format="iso")

equities  "stock market" lang:en       6533 posts  oldest=2026-08-13
equities  "S&P 500" lang:en            2182 posts  oldest=2026-08-13
macro     "the Fed" lang:en            5522 posts  oldest=2026-08-13
macro     Warsh lang:en                3256 posts  oldest=2026-08-13
macro     FOMC lang:en                  418 posts  oldest=2026-08-13
crypto    bitcoin lang:en             15788 posts  oldest=2026-08-13
crypto    ethereum lang:en             2623 posts  oldest=2026-08-13

--- funnel ---
rows returned by all queries : 36322
unique posts (by uri)        : 34533
posts in >1 stream           : 243
posts per stream             : {'crypto': 17534, 'macro': 8737, 'equities': 8507}
date range                   : 2026-08-13 00:00:16.084000+00:00 → 2026-09-12 23:59:24.870000+00:00


We got 36.322 rows from 7 different queries. When I cluster the data with uri we only have 34.533 unique posts. The oldest post dates are correct like my decision to start on 13th august and there is nothing after 13 september so we are okay.

After the volume checks and the check of the query texts we are confident with our data and can start our journey.

## 6. Reply threads
The search only returned the posts with my keywords in them, not the conversations under them, for the network I have to know who replied to who? So for every collected post that has at least one reply (8,908 posts) I fetch its thread with distinguishing who replied to whom  and the raw material of the response network is produced in this cell.

In [10]:
from atproto_client.exceptions import RequestException
import pickle

def walk_replies(node, parent_uri, parent_did, out, depth=1, max_depth=8):
    """Depth-first walk of a thread subtree; one row per reply post."""
    post = getattr(node, "post", None)
    if post is None or depth > max_depth:          # NotFoundPost / BlockedPost
        return
    out.append({
        "uri": post.uri, "author_did": post.author.did, "author_handle": post.author.handle,
        "created_at": parse_dt_utc(post.record.created_at), "text": post.record.text,
        "reply_count": post.reply_count, "like_count": post.like_count, "repost_count": post.repost_count,
        "langs": getattr(post.record, "langs", None),
        "parent_uri": parent_uri, "parent_did": parent_did, "depth": depth,
    })
    for child in getattr(node, "replies", None) or []:
        walk_replies(child, post.uri, post.author.did, out, depth + 1, max_depth)


targets = posts.loc[posts.reply_count > 0, ["uri", "author_did"]].reset_index(drop=True)
CKPT = f"{RAW_DIR}/threads_checkpoint.pkl"

# resume support: reload progress if the checkpoint exists
if os.path.exists(CKPT):
    thread_rows, failed, done = pickle.load(open(CKPT, "rb"))
else:
    thread_rows, failed, done = [], [], set()
print(f"threads to fetch: {len(targets)} | already done: {len(done)}")

for i, (uri, did) in enumerate(targets.itertuples(index=False), start=1):
    if uri in done:
        continue
    try:
        res = call_with_retries(client.app.bsky.feed.get_post_thread,
                                models.AppBskyFeedGetPostThread.Params(uri=uri, depth=8))
        for child in getattr(res.thread, "replies", None) or []:
            walk_replies(child, uri, did, thread_rows)
    except RequestException as e:                      # deleted / blocked / not found
        failed.append({"uri": uri, "error": type(e).__name__})
    done.add(uri)
    if i % 500 == 0 or i == len(targets):
        pickle.dump((thread_rows, failed, done), open(CKPT, "wb"))
        print(f"{i}/{len(targets)} threads | {len(thread_rows)} replies | {len(failed)} failed")
    time.sleep(0.2)

replies = pd.DataFrame(thread_rows).drop_duplicates("uri")
replies["collected_at"] = datetime.now(timezone.utc)
replies.to_pickle(f"{RAW_DIR}/replies_threads.pkl")
replies.to_json(f"{RAW_DIR}/replies_threads.jsonl", orient="records", lines=True, date_format="iso")
pd.DataFrame(failed).to_csv(f"{RAW_DIR}/threads_failed.csv", index=False)
print("done:", len(replies), "unique replies |", len(failed), "threads failed (deleted/blocked)")

threads to fetch: 8908 | already done: 0
500/8908 threads | 2516 replies | 0 failed
1000/8908 threads | 4652 replies | 0 failed
1500/8908 threads | 6969 replies | 0 failed
2000/8908 threads | 9096 replies | 0 failed
2500/8908 threads | 11142 replies | 0 failed
3000/8908 threads | 13752 replies | 0 failed
3500/8908 threads | 15941 replies | 0 failed
4000/8908 threads | 18118 replies | 0 failed
4500/8908 threads | 20493 replies | 0 failed
5000/8908 threads | 22362 replies | 0 failed
5500/8908 threads | 24379 replies | 0 failed
6000/8908 threads | 26769 replies | 0 failed
6500/8908 threads | 29695 replies | 0 failed
7000/8908 threads | 30416 replies | 0 failed
7500/8908 threads | 30549 replies | 0 failed
8000/8908 threads | 32564 replies | 0 failed
8500/8908 threads | 34149 replies | 0 failed
8908/8908 threads | 35648 replies | 0 failed
done: 28556 unique replies | 0 threads failed (deleted/blocked)


Here we have learned that in 34.533 posts 8.908 of them got at least 1 reply. There are a total of 35,648 raw response lines; when deduplicated by URI, this results in 28,556 responses (meaning 7,092 lines approximately 20% were duplicates). The raw count is higher because the subtree of a target post which is itself a response is traversed twice: once within its own thread and again within the root's thread.